# 09 · Regresión de verdad: cuándo una diferencia es real

**Módulo 2 · Datasets y experimentos** — *tiempo estimado: 80 minutos* — *consumo: 0 trazas en local*

Cambias el prompt. El acierto pasa del 72 % al 78 %. ¿Has mejorado el sistema?

**No lo sabes.** Y este notebook va de que la respuesta honesta casi siempre es esa,
hasta que mides el ruido.

Es el notebook con más matemáticas del curso, y la matemática es de bachillerato. Lo que
tiene de difícil es aceptar la conclusión.

Al terminar sabrás:

1. De dónde sale el ruido, que son **tres fuentes distintas**.
2. **Medirlo** en tu propio sistema, en vez de suponerlo.
3. Por qué un conjunto de 10 casos no sirve para nada, con el número delante.
4. Cuántas repeticiones y cuántos casos necesitas para el efecto que quieres detectar.
5. Escribir una **puerta de CI** que no dé falsas alarmas ni deje pasar regresiones.

In [ ]:
import sys, pathlib
sys.path.insert(0, str(next(p for p in [pathlib.Path.cwd(), *pathlib.Path.cwd().parents]
                            if (p / "utils" / "curso.py").exists())))

import math, random, statistics
from utils.curso import (init, online, cliente, separador, tickets,
                         ejemplos_locales, experimento_local, resumen_del_experimento)

init(silencioso=True)
print("listo")

## 1. Tres fuentes de ruido, y solo controlas una

| Fuente | De dónde viene | ¿La controlas? |
|---|---|---|
| **El modelo** | Muestreo, cambios del proveedor sin avisar, incluso con `temperature=0` | No |
| **El juez** | Es otro modelo. Tiene su propio ruido, encima del anterior | No |
| **La muestra** | Tus 30 casos son una muestra de todos los posibles | **Sí**: eligiéndola bien |

Las dos primeras son las que sorprenden. `temperature=0` **no da determinismo**: reduce
el muestreo pero no elimina la variación numérica del hardware, y no impide que el
proveedor cambie el modelo bajo el mismo nombre.

La tercera es la única sobre la que puedes actuar, y es la que este notebook mide.

## 2. Medir el ruido, en vez de suponerlo

El experimento es sencillo: **coger un sistema, ejecutarlo varias veces sobre el mismo
conjunto sin cambiar nada, y mirar cuánto bailan las notas**.

Para poder controlar el experimento, usamos un sistema cuyo acierto real conocemos: uno
que acierta con probabilidad `p`. Así sabemos la respuesta verdadera y podemos ver
cuánto se desvía la medida.

In [ ]:
def sistema_con_acierto(p: float, semilla: int):
    """Un sistema cuyo acierto REAL es `p`. Lo usamos para medir el ruido de la medida."""
    aleatorio = random.Random(semilla)

    def ejecutar(entradas: dict) -> dict:
        return {"acierta": aleatorio.random() < p}

    return ejecutar


def acierto(outputs: dict, reference_outputs: dict) -> dict:
    return {"key": "acierto", "score": float((outputs or {}).get("acierta", False))}


CONJUNTO_30 = ejemplos_locales(tickets(30), entradas=("asunto", "mensaje"),
                               salidas=("categoria",))

notas = [resumen_del_experimento(
            experimento_local(sistema_con_acierto(0.70, semilla), CONJUNTO_30,
                              evaluadores=[acierto]))["acierto"]
         for semilla in range(15)]

separador("15 ejecuciones del MISMO sistema, sobre el MISMO conjunto")
print("  notas:", [f"{n:.2f}" for n in notas])
print(f"\n  acierto real          : 0.70")
print(f"  media de las medidas  : {statistics.mean(notas):.3f}")
print(f"  desviación típica     : {statistics.stdev(notas):.3f}")
print(f"  la peor y la mejor    : {min(notas):.2f} y {max(notas):.2f}"
      f"  ->  {100 * (max(notas) - min(notas)):.0f} puntos de diferencia")

Ahí está el problema, en la última línea. **El mismo sistema, sin tocar nada, medido
quince veces, da notas que se llevan decenas de puntos.**

Si el lunes mides 0,63 y el viernes 0,77, y por el medio cambiaste el prompt, la
conclusión natural —«he mejorado 14 puntos»— puede ser sencillamente falsa: el sistema
puede no haber cambiado nada.

## 3. La ley que gobierna todo esto

El ruido no es caprichoso. Cuando mides una proporción sobre `n` casos, la desviación
típica de tu medida es:

```
σ = √( p·(1−p) / n )
```

Lo importante es el `n` en el denominador **dentro de una raíz**: para reducir el ruido
a la mitad, hace falta **cuatro veces más casos**.

Vamos a comprobar que la fórmula es cierta en tu máquina, no solo en el libro.

In [ ]:
separador("ruido medido frente a ruido teórico")
print(f"{'casos':>7} {'σ medida':>10} {'σ teórica':>11} {'peor-mejor':>12}")
print("-" * 44)

for n in (10, 30, 100, 200):
    conjunto = ejemplos_locales(tickets(n), entradas=("asunto", "mensaje"),
                                salidas=("categoria",))
    medidas = [resumen_del_experimento(
                  experimento_local(sistema_con_acierto(0.70, s), conjunto,
                                    evaluadores=[acierto]))["acierto"]
               for s in range(15)]
    teorica = math.sqrt(0.70 * 0.30 / n)
    print(f"{n:>7} {statistics.stdev(medidas):>10.4f} {teorica:>11.4f} "
          f"{max(medidas) - min(medidas):>11.2f}")

La medida sigue a la teoría muy de cerca. Y la última columna es la que hay que
interiorizar:

> **Con 10 casos, dos ejecuciones del mismo sistema difieren en 60 puntos de acierto.**
> No seis puntos: sesenta. El mismo sistema, sin tocar nada.

Por eso el notebook 06 decía que por debajo de 20-30 casos la diferencia entre dos
experimentos es ruido. Ahora está el número.

## 4. La calculadora que hay que usar antes de sacar conclusiones

Dale la vuelta a la fórmula: **¿cuántos casos necesito para detectar una mejora de X
puntos?**

Para comparar dos proporciones con una confianza razonable (95 %) y una probabilidad
decente de detectar el efecto si existe (80 %), la regla práctica es:

```
n ≈ 16 · p·(1−p) / efecto²
```

In [ ]:
def casos_necesarios(efecto: float, *, acierto_base: float = 0.7) -> int:
    """Casos por experimento para detectar una mejora de `efecto` (en proporción).

    Aproximación estándar para comparar dos proporciones al 95 % de confianza y 80 %
    de potencia. Sirve para lo que hay que usarla: para saber si tu conjunto está en
    el orden de magnitud correcto, no para publicar.
    """
    return math.ceil(16 * acierto_base * (1 - acierto_base) / (efecto ** 2))


separador("cuántos casos hacen falta")
print(f"{'mejora a detectar':>20} {'casos necesarios':>18}")
print("-" * 40)
for efecto in (0.20, 0.10, 0.05, 0.02, 0.01):
    print(f"{efecto:>19.0%} {casos_necesarios(efecto):>18,}")

Léelo despacio, porque cambia cómo se trabaja:

- Detectar **una mejora de 20 puntos** necesita unos 84 casos. Eso es factible.
- Detectar **5 puntos** necesita más de 1.300.
- Detectar **1 punto** necesita **más de 33.000**.

Con el plan Developer y sus 5.000 trazas al mes (notebook 00), **el conjunto que puedes
permitirte detecta mejoras grandes y nada más**.

Y eso no es una limitación del plan gratuito: es aritmética. Con 30 casos, cualquier
diferencia menor de unos 15 puntos que veas **es ruido hasta que se demuestre lo
contrario**.

> **La consecuencia práctica, y es liberadora:** deja de perseguir mejoras de dos
> puntos. No las puedes medir. Persigue cambios grandes, y para los pequeños usa otra
> cosa —comparación por pares (apartado 6) o mirar casos concretos—.

## 5. Repeticiones: para qué sirven y para qué no

`num_repetitions` ejecuta todo N veces. Sirve para **separar el ruido del modelo del
ruido de la muestra**, que son cosas distintas:

- **Repetir sobre el mismo conjunto** reduce el ruido del *modelo*. Los mismos casos,
  varias tiradas.
- **Ampliar el conjunto** reduce el ruido de la *muestra*.

Si tu sistema es determinista —un clasificador por reglas—, repetir **no sirve de nada**
y multiplica el coste por N. Compruébalo antes de pagarlo.

In [ ]:
def deteccion_de_no_determinismo(sistema, conjunto, *, repeticiones: int = 5) -> dict:
    """¿Merece la pena repetir? Compruébalo antes de multiplicar tu factura por N."""
    resultados = experimento_local(sistema, conjunto, evaluadores=[acierto],
                                   repeticiones=repeticiones)
    por_caso: dict[str, set[float]] = {}
    for fila in resultados:
        clave = str(fila["example"].id)
        for res in fila["evaluation_results"]["results"]:
            por_caso.setdefault(clave, set()).add(res.score)

    inestables = [c for c, puntuaciones in por_caso.items() if len(puntuaciones) > 1]
    return {"casos": len(por_caso), "inestables": len(inestables),
            "veredicto": ("repetir SÍ aporta" if inestables
                          else "determinista: repetir solo multiplica el coste")}


def clasificador_determinista(entradas: dict) -> dict:
    texto = f"{entradas.get('asunto', '')} {entradas.get('mensaje', '')}".lower()
    return {"acierta": "factur" in texto or "cobr" in texto}


CONJUNTO_20 = ejemplos_locales(tickets(20), entradas=("asunto", "mensaje"),
                               salidas=("categoria",))

print("sistema determinista :", deteccion_de_no_determinismo(clasificador_determinista,
                                                             CONJUNTO_20))
print("sistema con azar     :", deteccion_de_no_determinismo(sistema_con_acierto(0.7, 1),
                                                             CONJUNTO_20))

La primera línea es dinero ahorrado: si tu sistema da lo mismo en las cinco pasadas,
`num_repetitions=3` te está costando el triple para no añadir información.

**Ojo con un matiz:** eso vale para el sistema. Si tu **juez** es un LLM, el juez tiene
su propio no-determinismo aunque el sistema sea determinista, y ahí repetir sí aporta.
Este chequeo mide los dos juntos, que es lo que importa.

## 6. La puerta de la CI: qué comparar y contra qué

Con todo lo anterior, la regla de decisión. Y no es «¿subió la media?».

In [ ]:
def decidir(nota_nueva: float, nota_anterior: float, *, n: int,
            tolerancia_sigmas: float = 2.0) -> tuple[str, str]:
    """Compara dos experimentos teniendo en cuenta el ruido de la medida.

    La tolerancia se calcula, no se inventa: `sigmas` desviaciones típicas del ruido
    esperado para ese tamaño de conjunto. Con 2σ, una diferencia dentro de la banda
    tiene una explicación de sobra en el azar.
    """
    p = (nota_nueva + nota_anterior) / 2
    sigma_diferencia = math.sqrt(2 * p * (1 - p) / n)
    banda = tolerancia_sigmas * sigma_diferencia
    diferencia = nota_nueva - nota_anterior

    if diferencia < -banda:
        return "BLOQUEA", f"regresión de {abs(diferencia):.1%} (banda de ruido ±{banda:.1%})"
    if diferencia > banda:
        return "MEJORA", f"mejora de {diferencia:.1%} por encima del ruido (±{banda:.1%})"
    return "RUIDO", f"diferencia de {diferencia:+.1%} dentro de la banda ±{banda:.1%}"


separador("la misma diferencia, con distintos tamaños de conjunto")
for n in (20, 100, 1000):
    veredicto, razon = decidir(0.78, 0.72, n=n)
    print(f"  n={n:>5}: 72 % -> 78 %   {veredicto:<8} {razon}")

**La misma diferencia de seis puntos es ruido con 20 casos y una mejora real con 1.000.**
Esa es toda la idea del notebook, en una tabla.

Y ahora la parte incómoda: la mayoría de los equipos deciden con la primera fila y actúan
como si fuera la tercera.

In [ ]:
separador("la puerta completa de la CI")

def puerta_de_ci(resultados_nuevos, nota_anterior: float, *, esperadas: set[str],
                 tolerancia_sigmas: float = 2.0) -> tuple[bool, list[str]]:
    """Junta las comprobaciones del notebook 07 con la banda de ruido de este.

    El orden importa: primero se comprueba que la MEDIDA es válida, y solo después se
    mira el número. Una nota estupenda sobre una medida rota no es una buena noticia.
    """
    filas = list(resultados_nuevos)
    medias = resumen_del_experimento(resultados_nuevos)
    problemas = []

    faltan = esperadas - {k.replace(" (resumen)", "") for k in medias}
    if faltan:
        problemas.append(f"métricas ausentes (¿se cayó el juez?): {sorted(faltan)}")

    reventados = sum(1 for f in filas if f["run"].error)
    if reventados:
        problemas.append(f"{reventados} caso(s) reventaron")

    puntuados = sum(1 for f in filas
                    for r in f["evaluation_results"]["results"]
                    if r.key == "acierto" and r.score is not None)
    if puntuados < len(filas):
        problemas.append(f"cobertura {puntuados}/{len(filas)}: la media es de otro conjunto")

    if problemas:
        return False, problemas          # la medida no vale: no se mira el número

    veredicto, razon = decidir(medias["acierto"], nota_anterior,
                               n=len(filas), tolerancia_sigmas=tolerancia_sigmas)
    return veredicto != "BLOQUEA", [f"{veredicto}: {razon}"]


for etiqueta, sistema, anterior in [
    ("igual que antes     ", sistema_con_acierto(0.70, 99), 0.70),
    ("mejora pequeña      ", sistema_con_acierto(0.76, 99), 0.70),
    ("regresión grande    ", sistema_con_acierto(0.40, 99), 0.70),
]:
    resultados = experimento_local(sistema, CONJUNTO_30, evaluadores=[acierto])
    pasa, mensajes = puerta_de_ci(resultados, anterior, esperadas={"acierto"})
    print(f"  {etiqueta} {'PASA ' if pasa else 'BLOQUEA'}  {mensajes[0]}")

La segunda fila es la interesante: el sistema es de verdad seis puntos mejor, la medida
dice trece, y aun así **la puerta lo declara ruido**. Sobre 30 casos la banda es de más
de veinte puntos, así que no hay forma de distinguirlo del azar.

Y está bien que sea así. La puerta no dice «no has mejorado»: dice «con este conjunto no
lo puedo saber», que es la verdad.

Lo que la puerta hace es evitar los dos errores caros:

- **Celebrar ruido**, que lleva a meter cambios que no mejoran nada y a perder la
  capacidad de saber qué funcionó.
- **Bloquear por ruido**, que lleva a que la gente desactive la puerta.

## 7. Comparación por pares: cuando no hay respuesta correcta

Todo lo anterior supone que existe una referencia con la que comparar. Para texto libre
—una redacción, un resumen, una explicación— muchas veces no la hay.

La alternativa es **preguntar cuál de las dos es mejor**, que es una pregunta mucho más
fácil que «¿cuánto de buena es esta?». `evaluate_comparative` hace eso sobre dos
experimentos ya ejecutados.

In [ ]:
import inspect
from langsmith.evaluation import evaluate_comparative   # ojo: no está en el nivel superior

for nombre, p in inspect.signature(evaluate_comparative).parameters.items():
    print(f"  {nombre:<20} = {p.default!r}")

Fíjate en **`randomize_order`**. Es el arreglo directo del **sesgo de posición** del
notebook 08: un juez tiende a preferir una de las dos posiciones, así que si siempre
pones el sistema nuevo el segundo, medirás esa preferencia y la llamarás mejora.

**Ponlo a `True` siempre.** Es una palabra y elimina un sesgo entero.

In [ ]:
@online("Comparar dos experimentos por pares", trazas=0)
def _():
    """Toma dos experimentos YA EJECUTADOS, así que no vuelve a pagar las ejecuciones."""
    def cual_es_mejor(inputs: dict, outputs: list[dict]) -> dict:
        """Devuelve las puntuaciones de las dos, en el mismo orden que `outputs`.

        Con un juez de verdad, aquí iría la llamada. La respuesta que se le pide es
        «¿cuál de estas dos responde mejor?», no «puntúa cada una».
        """
        return {"key": "preferencia", "scores": [1, 0]}

    evaluate_comparative(
        ("clasificador-v1", "clasificador-v2"),   # nombres de los experimentos
        evaluators=[cual_es_mejor],
        randomize_order=True,                     # <- el arreglo del sesgo de posición
        client=cliente(),
    )

## 8. Y lo que invalida todo: comparar contra otro conjunto

Última pieza, y la más fácil de pisar. Todo este notebook supone que los dos experimentos
corrieron **sobre exactamente los mismos casos**.

Si por el medio alguien añadió diez ejemplos difíciles al dataset, tu «regresión» es que
el examen se puso más difícil. Y no hay ninguna señal que te avise.

Por eso el notebook 06 insistía en **fijar la versión** con `as_of`, y por eso conviene
que la puerta de la CI lo compruebe:

In [ ]:
def misma_base(experimento_a: dict, experimento_b: dict) -> list[str]:
    """Comprueba que dos experimentos son comparables ANTES de comparar sus números."""
    problemas = []
    if experimento_a["version_dataset"] != experimento_b["version_dataset"]:
        problemas.append(f"versiones distintas del dataset: "
                         f"{experimento_a['version_dataset']} vs {experimento_b['version_dataset']}")
    if experimento_a["n_casos"] != experimento_b["n_casos"]:
        problemas.append(f"distinto número de casos: "
                         f"{experimento_a['n_casos']} vs {experimento_b['n_casos']}")
    if experimento_a["evaluadores"] != experimento_b["evaluadores"]:
        problemas.append("los evaluadores no son los mismos")
    return problemas


anterior = {"version_dataset": "v1", "n_casos": 30, "evaluadores": ["acierto", "f1"]}
nuevo_ok = {"version_dataset": "v1", "n_casos": 30, "evaluadores": ["acierto", "f1"]}
nuevo_mal = {"version_dataset": "v2", "n_casos": 42, "evaluadores": ["acierto"]}

print("comparables :", misma_base(anterior, nuevo_ok) or "sí")
print("comparables :", misma_base(anterior, nuevo_mal) or "sí")

## 9. Ejercicios

### Ejercicio 1 — Tu propia tabla de sensibilidad

Escribe `sensibilidad(n, repeticiones)` que devuelva **la mejora más pequeña que puedes
detectar** con ese conjunto y esas repeticiones. Úsala para responder: con 5.000 trazas
al mes y un juez LLM, ¿cuál es la mejora más pequeña que puedes medir una vez al mes?

In [ ]:
# Tu solución aquí.

<details>
<summary><b>Solución</b></summary>

In [ ]:
def sensibilidad(n: int, repeticiones: int = 1, *, acierto_base: float = 0.7,
                 sigmas: float = 2.0) -> float:
    """La mejora más pequeña que este experimento puede distinguir del ruido.

    Repetir reduce el ruido del modelo, así que el tamaño efectivo es n × repeticiones.
    """
    efectivo = n * repeticiones
    sigma_diferencia = math.sqrt(2 * acierto_base * (1 - acierto_base) / efectivo)
    return sigmas * sigma_diferencia


separador("qué puedes detectar con tu presupuesto")
print(f"{'casos':>7}{'reps':>6}{'trazas':>9}{'detecta desde':>16}")
print("-" * 40)

for n, reps in [(20, 1), (30, 1), (30, 3), (50, 1), (100, 1), (250, 1), (1000, 1)]:
    trazas = n * reps * 2          # una del sistema y una del juez, por caso
    print(f"{n:>7}{reps:>6}{trazas:>9,}{sensibilidad(n, reps):>15.1%}")

print()
PRESUPUESTO = 5000
for reps in (1, 3):
    n = PRESUPUESTO // (2 * reps)
    print(f"  {reps} repetición(es): {n:>5} casos  ->  {n * reps:>5} medidas  "
          f"->  detectas desde {sensibilidad(n, reps):.1%}")

Con todo el presupuesto mensual gastado de una vez llegas a detectar mejoras de unos
dos puntos y medio. Ni un punto, y habiéndote fundido el mes entero en una sola medida.

Y fíjate en que las dos últimas líneas dan **exactamente el mismo número**: no es un
fallo, es la moraleja. Lo que manda es el total de medidas —`casos × repeticiones`—, y
con el mismo presupuesto sale igual repartirlo de una forma o de otra. Lo que cambia es
**cuál de los dos ruidos reduces**, el del modelo o el de la muestra, y eso lo decide el
chequeo del apartado 5.

Lo que ninguna de las dos opciones te da es medir mejoras de un punto. Para eso no hay
presupuesto: hacen falta cientos de miles de casos.

</details>

### Ejercicio 2 — Cuánto te engañarías sin la banda

Simula un año de trabajo: **veinticuatro cambios quincenales que no mejoran nada**
—el sistema es idéntico— y cuenta cuántas veces habrías declarado «mejora» mirando solo
la media, frente a cuántas con la puerta del apartado 6.

In [ ]:
# Tu solución aquí.

<details>
<summary><b>Solución</b></summary>

In [ ]:
def simular_un_ano(n_casos: int, *, cambios: int = 24, acierto_real: float = 0.70):
    """Todos los cambios son cosméticos: el acierto real NO cambia nunca."""
    conjunto = ejemplos_locales(tickets(n_casos), entradas=("asunto", "mensaje"),
                               salidas=("categoria",))

    notas = [resumen_del_experimento(
                experimento_local(sistema_con_acierto(acierto_real, s), conjunto,
                                  evaluadores=[acierto]))["acierto"]
             for s in range(cambios + 1)]

    falsas_mejoras = falsas_regresiones = con_banda = 0
    for anterior, nueva in zip(notas, notas[1:]):
        if nueva > anterior:
            falsas_mejoras += 1
        elif nueva < anterior:
            falsas_regresiones += 1
        if decidir(nueva, anterior, n=n_casos)[0] != "RUIDO":
            con_banda += 1
    return notas, falsas_mejoras, falsas_regresiones, con_banda


separador("un año de cambios que no cambian nada")
for n_casos in (20, 100):
    notas, mejoras, regresiones, con_banda = simular_un_ano(n_casos)
    print(f"\n  conjunto de {n_casos} casos (acierto real siempre 70 %)")
    print(f"    notas observadas: {min(notas):.0%} a {max(notas):.0%}")
    print(f"    mirando solo la media -> {mejoras} «mejoras» y {regresiones} «regresiones» "
          f"de {len(notas) - 1} cambios")
    print(f"    con la banda de ruido -> {con_banda} conclusión(es)")

Veinticuatro cambios que no mejoran nada, y mirando solo la media habrías escrito una
veintena de notas de versión celebrando mejoras y lamentando regresiones. Ninguna era
real.

Con la banda, casi todas se clasifican correctamente como ruido.

**Y aquí está el coste verdadero de no tener banda**, que no es escribir notas de versión
equivocadas: es que **el ruido tapa las regresiones de verdad**. Cuando tu registro está
lleno de subidas y bajadas aleatorias, la caída real del día que rompiste algo se pierde
entre ellas, y nadie la mira.

Una puerta con banda tiene la propiedad contraria: **casi nunca dice nada, y cuando dice
algo hay que hacerle caso.** Que es exactamente lo que quieres de una alarma.

</details>

## 10. Resumen

- El ruido tiene **tres fuentes** —modelo, juez, muestra— y solo controlas la tercera.
  `temperature=0` no da determinismo.
- **Mídelo, no lo supongas**: ejecuta el mismo sistema varias veces y mira el rango. Con
  10 casos, dos ejecuciones idénticas pueden diferir en **50 puntos**.
- `σ = √(p(1−p)/n)`, comprobado contra la medida. Para reducir el ruido a la mitad
  necesitas **cuatro veces más casos**.
- Detectar 20 puntos: ~84 casos. Detectar 5: más de 1.300. Detectar 1: más de 33.000.
  **Con un presupuesto normal solo puedes medir mejoras grandes**, y eso es aritmética,
  no una limitación del plan.
- **Comprueba si tu sistema es determinista** antes de pagar repeticiones. Si lo es, `N`
  repeticiones son `N` veces el coste y cero información — salvo que el juez sea un LLM.
- La puerta de la CI compara contra una **banda calculada**, no contra un umbral fijo. Y
  primero valida la medida (cobertura, métricas presentes, casos reventados) y solo
  después mira el número.
- `randomize_order=True` en `evaluate_comparative` elimina el sesgo de posición del juez.
- Y todo lo anterior **se cae si los dos experimentos no corrieron sobre la misma versión
  del dataset**. Compruébalo antes de comparar.

**Siguiente:** [`10_pruebas_con_modelo_en_ci`](10_pruebas_con_modelo_en_ci.ipynb) — cómo
tener pruebas con LLM que sean deterministas, gratis y versionadas, que es lo que el
curso de LangGraph deja abierto.